In [1]:
import pandas as pd

# DUPLICATE REMOVAL


## What is a Duplicate?

A duplicate is a record that appears more than once
in a dataset when it represents the same observation
or real-world entity.

Example:

| ID | Name | Age | City |
|---:|---|---:|---|
| 101 | Arun | 25 | Chennai |
| 102 | Priya | 24 | Madurai |
| 101 | Arun | 25 | Chennai |

The first and third rows represent the same record,
so one of them may be a duplicate.

---

## Why Remove Duplicates?

Duplicates can:

- Distort statistical calculations
- Affect counts and percentages
- Give duplicated observations more weight
- Affect machine learning models
- Increase the size of the dataset unnecessarily

---

## Important

Do NOT remove repeated rows blindly.

A repeated value does not always mean a duplicate.

Example:

| Customer | Product | Date |
|---|---|---|
| Arun | Laptop | Jan 1 |
| Arun | Laptop | Feb 1 |

These may be two different transactions.

Always ask:

"Do these rows represent the same real-world observation?"

---

## Duplicate Removal Workflow

Raw Dataset
    ↓
Detect duplicates
    ↓
Understand why they exist
    ↓
Determine whether they are true duplicates
    ↓
Remove duplicates if appropriate
    ↓
Verify the result

---

## Pandas Functions

### Detect duplicates

```python
df.duplicated()

In [3]:
import pandas as pd

data = {
    "Name": ["Arun", "Priya", "Arun", "Karthik"],
    "Age": [25, 24, 25, 30],
    "City": ["Chennai", "Madurai", "Chennai", "Coimbatore"]
}

df = pd.DataFrame(data)

df

,Name,Age,City
0,Arun,25,Chennai
1,Priya,24,Madurai
2,Arun,25,Chennai
3,Karthik,30,Coimbatore


In [4]:
df.duplicated()

0    False
1    False
2     True
3    False
dtype: bool

In [5]:
df.duplicated().sum()

np.int64(1)

In [6]:
df = df.drop_duplicates()

In [7]:
df

,Name,Age,City
0,Arun,25,Chennai
1,Priya,24,Madurai
3,Karthik,30,Coimbatore


# EXACT DUPLICATES

## Definition

An exact duplicate occurs when every value in two or more
rows is exactly the same.

Example:

| ID | Name | Age | City |
|---:|---|---:|---|
| 101 | Arun | 25 | Chennai |
| 102 | Priya | 24 | Madurai |
| 101 | Arun | 25 | Chennai |

Rows with ID 101 are exact duplicates because every
column has the same value.

---

## Detect Exact Duplicates

```python
df.duplicated()

In [8]:
data = {
    "Name": ["Arun", "Priya", "Arun", "Karthik"],
    "Age": [25, 24, 25, 30],
    "City": ["Chennai", "Madurai", "Chennai", "Coimbatore"]
}

df = pd.DataFrame(data)

df

,Name,Age,City
0,Arun,25,Chennai
1,Priya,24,Madurai
2,Arun,25,Chennai
3,Karthik,30,Coimbatore


In [9]:
df.duplicated()

0    False
1    False
2     True
3    False
dtype: bool

In [10]:
# If u wanna just see actual duplicated rows
df[df.duplicated()]

,Name,Age,City
2,Arun,25,Chennai


In [11]:
# if u wanna see the duplicates with index
df[df.duplicated(keep= False)]

,Name,Age,City
0,Arun,25,Chennai
2,Arun,25,Chennai


In [12]:
# last occurence concidered as original
df[df.duplicated(keep="last")]

,Name,Age,City
0,Arun,25,Chennai


In [14]:
# first occurence concidered as original
# so its returns last as duplicated
df[df.duplicated(keep="first")]

,Name,Age,City
2,Arun,25,Chennai


In [15]:
df = df.drop_duplicates()
df

,Name,Age,City
0,Arun,25,Chennai
1,Priya,24,Madurai
3,Karthik,30,Coimbatore


# PARTIAL DUPLICATES

## Definition

A partial duplicate occurs when two or more rows have
the same important identifying columns, while one or
more other columns are different.

Unlike exact duplicates, not every column has to match.

---

## Example

| Customer_ID | Name | City | Amount |
|---:|---|---|---:|
| 101 | Arun | Chennai | 500 |
| 102 | Priya | Madurai | 700 |
| 101 | Arun | Chennai | 800 |

Rows with Customer_ID 101 may represent the same
customer, even though Amount is different.

Therefore, they are not exact duplicates, but may
be partial duplicates.

---

## Detect Partial Duplicates

Use the `subset` parameter:

```python
df.duplicated(subset=["Customer_ID"])

In [30]:
data = {
    "Customer_ID": [101, 102, 101, 103],
    "Name": ["Arun", "Priya", "Arun", "Kumar"],
    "City": ["Chennai", "Madurai", "Chennai", "Trichy"],
    "Amount": [500, 700, 800, 300]
}

df = pd.DataFrame(data)

In [31]:
df

,Customer_ID,Name,City,Amount
0,101,Arun,Chennai,500
1,102,Priya,Madurai,700
2,101,Arun,Chennai,800
3,103,Kumar,Trichy,300


In [32]:
# customer id 101 everything is  same exept amount
# sometimes in duplicates we need give priority to customer_id alone
# in those cases we can use subset
# to identify the duplicates

In [33]:
df.duplicated(subset=["Customer_ID"])

0    False
1    False
2     True
3    False
dtype: bool

- sometimes the same customer may bought different product
- so we should not delete anythinng to remove the duplicates
- we need another things to use as subset like transaction id


| Customer_ID | Product  | Amount |
| ----------: | -------- | -----: |
|         101 | Laptop   |  50000 |
|         101 | Mouse    |   1000 |
|         101 | Keyboard |   2000 |


In [37]:
df = df.drop_duplicates(subset=["Customer_ID"],keep = 'last')
# this will keep last one as original
# by default(without last) it will keep first one as original

In [38]:
df

,Customer_ID,Name,City,Amount
1,102,Priya,Madurai,700
2,101,Arun,Chennai,800
3,103,Kumar,Trichy,300


# DUPLICATE DETECTION STRATEGIES



## 1. Entire Row

Check whether the complete row is duplicated.

```python
df.duplicated()

Count duplicates:

df.duplicated().sum()

Use when all columns should match for a duplicate.

2. Unique Identifier

If a column should uniquely identify a record:

df["Customer_ID"].duplicated().sum()

View all repeated IDs:

df[df["Customer_ID"].duplicated(keep=False)]

IMPORTANT:
A repeated ID is not automatically a duplicate.
It depends on the dataset.

Example:

Customer table:
Customer_ID should usually be unique.

Transaction table:
Customer_ID can appear multiple times because
one customer can make multiple transactions.

3. Multiple Identifying Columns

Use subset when several columns together
identify a record.

df.duplicated(
    subset=["Name", "Age", "City"]
)

Rows are considered duplicates when the selected
columns match.

4. Normalize Before Detection

Different formatting can hide duplicates.

Example:

Arun Kumar
ARUN KUMAR
arun kumar

Standardize:

df["Name"] = df["Name"].str.strip().str.lower()

Now the values become:

arun kumar
arun kumar
arun kumar

5. Remove Extra Spaces
df["Name"] = df["Name"].str.strip()

Useful for values such as:

" Arun"
"Arun "
" Arun "

6. Standardize Case
df["City"] = df["City"].str.lower()

Example:

Chennai
CHENNAI
chennai

becomes:

chennai
chennai
chennai


7. Investigate Before Removing

Recommended workflow:

Detect duplicates
Count duplicates
Display duplicate records
Understand why they are duplicated
Determine which columns identify a record
Decide whether duplicates are legitimate
Remove duplicates only when appropriate
Verify the result


8. Keep the Correct Record

Sometimes duplicate records contain different information.

Example:

Customer_ID	Phone	Updated_Date
101	98765	Jan 1
101	99999	Feb 1

If the latest record is considered correct:

df = df.sort_values("Updated_Date")

df = df.drop_duplicates(
    subset=["Customer_ID"],
    keep="last"
)


9. Duplicate Percentage
duplicate_count = df.duplicated().sum()
total_rows = len(df)

duplicate_percentage = (
    duplicate_count / total_rows * 100
)

In [39]:
data = {
    "Customer_ID": [101, 102, 101, 103, 102],
    "Name": [
        "Arun Kumar",
        "Priya",
        "ARUN KUMAR",
        "Karthik",
        "Priya"
    ],
    "City": [
        "Chennai",
        "Madurai",
        "CHENNAI",
        "Trichy",
        "Madurai"
    ],
    "Amount": [500, 700, 800, 300, 900]
}

df = pd.DataFrame(data)

In [41]:
df['Name'] = df['Name'].str.strip().str.lower()

In [42]:
df['City'] = df['City'].str.strip().str.lower()

In [43]:
df

,Customer_ID,Name,City,Amount
0,101,arun kumar,chennai,500
1,102,priya,madurai,700
2,101,arun kumar,chennai,800
3,103,karthik,trichy,300
4,102,priya,madurai,900


In [47]:
df[df.duplicated(subset=['Customer_ID','Name','City'],keep = False)]

,Customer_ID,Name,City,Amount
0,101,arun kumar,chennai,500
1,102,priya,madurai,700
2,101,arun kumar,chennai,800
4,102,priya,madurai,900
